# <div align="center" style="color:#EBF4F6; font-family:'Segoe UI', sans-serif;">Lab 4</div>
# <div align="center" style="color:#EBF4F6; font-family:'Segoe UI', sans-serif;">Assignment 3</div>

 * **Ahmed M. Y. Farani** | `1320236338`

 This is one of a series of Colab notebooks created for the **IR in Arabic** course. It demonstrates how we can employ query expansion to improve ranked retrieval and evaluate the output of multiple retrieval models.

The **learning outcomes** of the this notebook are:


*   Apply query expansion (**QE**) using the pseudo-relevance feedback (**PRF**) approach (with **RM3** as the term selection model).
*   Evaluate and compare the results of retrieval models with and without QE.
* Study the effect of parameter settings with QE.

### **Setup**

We will first install PyTerrier as follows:

In [ ]:
#install the PyTerrier framework
!pip install python-terrier

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 135.8 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=35192503c4208a5cc80d21213cc5d364be98d03d88b647e6af468ef954898683
  Stored in directory: /root/.

Since we need to apply QE, we need to install an additional PyTerrier package for pseudo-relevance feedback (**PRF**).

In [ ]:
# Need to install additional terrier package for PRF. It will take around 1 min
!git clone https://github.com/terrierteam/terrier-prf/
!apt-get install maven
%cd /content/terrier-prf/
!mvn install
!pwd
%cd ..

Cloning into 'terrier-prf'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 227 (delta 62), reused 192 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (227/227), 33.94 KiB | 589.00 KiB/s, done.
Resolving deltas: 100% (62/62), done.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaopalliance-java libapache-pom-java libatinject-jsr330-api-java
  libcdi-api-java libcommons-cli-java libcommons-io-java libcommons-lang3-java
  libcommons-parent-java libgeronimo-annotation-1.3-spec-java
  libgeronimo-interceptor-3.0-spec-java libguava-java libguice-java
  libhawtjni-runtime-java libjansi-java libjansi-native-java libjsr305-java
  libmaven-parent-java libmaven-resolver-java libmaven-shared-utils-java
  libmaven3-core-java libplexus-cipher-java libplexus-classworlds-java
  li

The next step is to initialise PyTerrier. This is performed using PyTerrier's init() method. The init() method is needed as PyTerrier must download Terrier's jar file and start the Java virtual machine. We prevent init() from being called more than once by checking started().

In [ ]:
import pyterrier as pt
if not pt.started():
  pt.init()

terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


/tmp/ipykernel_845/1742394333.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_845/1742394333.py:3: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


Another library that we need for this lab is Arabic-Stopwords

In [ ]:
#install the Arabic stop words library
!pip install Arabic-Stopwords

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 12.4 MB/s eta 0:00:00


We will import all the python libraries needed for this lab

In [ ]:
#we need to import the following libraries.
import pandas as pd
#to display the full text on the notebook without truncation
pd.set_option('display.max_colwidth', 150)
import numpy as np
import re
from snowballstemmer import stemmer
from tqdm import tqdm
import arabicstopwords.arabicstopwords as stp

### **Loading and Preparing Data**

We will prepare our helper functions for removing stop words, normalize, and stemming which we will use to process our queries.

**Reminder**: We need to apply the same preprocessing here that we applied when indexing our **EveTAR** dataset in previous labs.

In [ ]:
#removing Stop Words function
def remove_stop_words(sentence):
    terms=[]
    stopWords= set(stp.stopwords_list())
    for term in sentence.split() :
        if term not in stopWords :
           terms.append(term)
    return " ".join(terms)

#a function to normalize the tweets
def normalize(text):
    text = re.sub("[إأٱآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "ء", text)
    text = re.sub("ئ", "ء", text)
    text = re.sub("ة", "ه", text)
    return(text)

#define the stemming function
ar_stemmer = stemmer("arabic")
def stem(sentence):
    return " ".join([ar_stemmer.stemWord(i) for i in sentence.split()])


# perform all preprocessing steps
def preprocess(sentence):
  # apply preprocessing steps on the given sentence
  sentence = remove_stop_words(sentence)
  sentence = normalize(sentence)
  sentence = stem(sentence)
  return sentence

We will use our indexed **EveTAR** dataset. The index is uploaded in our Github repository so we will access it as follows:

In [ ]:
%rm -rf IR-in-Arabic
%rm -rf evetarIndex
!git clone https://github.com/nourfmohammed1/data_EveTAR.git
!unzip data_EveTAR/evetarIndex.zip -d evetarIndex
!ls evetarIndex

Cloning into 'data_EveTAR'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 26 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 5.81 MiB | 9.52 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Archive:  data_EveTAR/evetarIndex.zip
   creating: evetarIndex/evetarIndex/
  inflating: evetarIndex/evetarIndex/data.direct.bf  
  inflating: evetarIndex/evetarIndex/data.properties  
  inflating: evetarIndex/evetarIndex/data.lexicon.fsomaphash  
  inflating: evetarIndex/evetarIndex/data.lexicon.fsomapfile  
  inflating: evetarIndex/evetarIndex/data.inverted.bf  
  inflating: evetarIndex/evetarIndex/data.meta.zdata  
  inflating: evetarIndex/evetarIndex/data.meta.idx  
  inflating: evetarIndex/evetarIndex/data.document.fsarrayfile  
  inflating: evetarIndex/evetarIndex/data.meta-0.fsomapfile  
  inflating: evetarIndex/evetarIndex/data.lexicon.fsomapid  
evet

Next, we will load our index.

In [ ]:
#we will load the index
index_ref = pt.IndexRef.of("./evetarIndex/evetarIndex/data.properties")
index = pt.IndexFactory.of(index_ref)

Let's load our collection. We will later use it to read the full content of documents we retrieved from the index.

In [ ]:
dataset_links=["https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/evetar-q-01.txt",
               "https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/evetar-q-02.txt",
               "https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/evetar-q-03.txt",
               "https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/evetar-q-04.txt",
               "https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/evetar-q-05.txt",
               "https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/evetar-q-06.txt",
               "https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/evetar-q-07.txt",
               "https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/evetar-q-08.txt",
               "https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/evetar-q-09.txt",
               "https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/evetar-q-10.txt"]

full_data=pd.DataFrame()
for i in tqdm(range(len(dataset_links))):
    tweets=pd.read_csv(dataset_links[i], sep='\t')
    full_data=pd.concat([full_data,tweets],ignore_index=True)
full_data.reset_index(inplace=True,drop=True)
#the docno will be our tweetID
full_data["docno"]=full_data["tweetID"].astype(str)

100%|██████████| 10/10 [00:04<00:00,  2.15it/s]


We will load queries that are already defined and released with EveTAR dataset and process them using the same processing steps done when indexing EveTAR.

In [ ]:
#read the queries file from Github
topics=pd.read_csv("https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/topics.txt", sep='\t',names=['data'])
queries=[]
qid=[]
#we will get the queries and their ids from the topics file
for i in range(len(topics)):
    splitted=topics["data"][i:i+1][i].split(" ")
    if splitted[0]=="<title>":
       queries.append(' '.join(splitted[1:]))
    if splitted[0]=="<num>":
       qid.append(splitted[2])
queriesDF=pd.DataFrame()
#the queries datframe should have qid and query columns to retrieve using PyTerrier
queriesDF["qid"]=qid
queriesDF["raw_query"]=queries
#remove the stopwords from queries, do normalization, and apply stemming
queriesDF["query"]=queriesDF["raw_query"].apply(preprocess)

We now load qrels (relevance judgments) so that we can evaluate our runs.

In [ ]:
# Loading EveTAR qrels
qrels=pd.read_csv("https://raw.githubusercontent.com/nourfmohammed1/data_EveTAR/refs/heads/main/qrels.txt", sep='\t',names=['qid','Q0','docno','label'])
qrels['docno']=qrels['docno'].astype(str)
qrels = qrels[qrels["docno"].isin(full_data["docno"].tolist())]

### **Experiments with PRF for QE**

Once we have all our data ready, now we can see how to query expansion using PRF process.

**Experiment 1: query expansion using RM3**

We will use BatchRetrieve PyTerrier class for retrieval using BM25 model. We will then apply query expansion using RM3 model and re-run the retrieval to see the effect of expansion.

**What is RM3?**

A relevance modelling (RM) approach to represent the topic described by the input query. RM3 benifits from a set of documents retrieved in response to that query, to generate a relevance model from which we can compute the probability of observing a term from these documents. Recall that the input query is one way to represent a topic, other representative terms can co-occur with these query terms.

In [ ]:
# Define our retrieval model
bm25 = pt.BatchRetrieve(index, wmodel="BM25",num_results=1000)

/tmp/ipykernel_845/3990366921.py:2: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25 = pt.BatchRetrieve(index, wmodel="BM25",num_results=1000)


Let's try a single query using BM25 as in earlier labs.

In [ ]:
query="هجوم انتحاري على مقهى في طرابلس في لبنان"

#we need to process the query also as we did for documents
query = preprocess(query)

#we will call the search function using our retrieval model we set up above
results_noqe = bm25.search(query)
if len(results_noqe)==0:
   print("There are no relevant documents for your selected query.")
else:
   print(results_noqe[['docid','score']][0:5])
#Let's check the tweets text for the top 5 retrieved tweets
full_data[['tweetText']][full_data['docno'].isin(results_noqe['docno'].loc[0:5].tolist())]

   docid      score
0  21590  25.403218
1  22944  25.297453
2  22646  23.988924
3  22692  23.988924
4  22438  23.656683


,tweetText
21590,#لبنان - مقتل 7 في هجوم انتحاري مزدوج على مقهى في #طرابلس #bahrain #lebanon
22438,هجوم انتحاري علي مقهي في لبنان http://t.co/06f2Npw9Zg
22466,لبنان : مقتل تسعة أشخاص و 35 جريحاً في هجوم انتحاري استهدف مقهى بطرابلس http://t.co/48WLGUQm20 http://t.co/vSdXUlb0tH
22646,مقتل 7 في هجوم انتحاري على مقهى في لبنان: أدى هجوم انتحاري على مقهى في مدينة طرابلس اللبنانية إلى مقتل سبعة أش... http://t.co/d1zWvCG3dh
22692,مقتل 7 في هجوم انتحاري على مقهى في لبنان: أدى هجوم انتحاري على مقهى في مدينة طرابلس اللبنانية إلى مقتل سبعة أش... http://t.co/uwvvF6V5bD
22944,مقتل سبعة على الأقل في هجوم انتحاري على مقهى في لبنان - أدى هجوم انتحاري على مقهى في مدينة طرابلس اللبنانية إلى ... http://t.co/kRD9oPXlra


Let's now see the effect of query expansion on the query above. The process we will perform is as follows:


1. Retrieve an initial set of "psuedo-relevant documents" using a retrieval model (e.g., BM25).
2. Use **RM3** method to score terms in these documents
3. Expand the current query with top terms from Step 2
4. Retrieve the final set of documents with the expanded query.



In [ ]:
# "rewrite" function from PyTerrier will be used to expand queries specifying RM3 as the model
# fb_docs ==> no. expansion documents
# fb_terms ==> no. expansion terms
rm3_expander = pt.rewrite.RM3(index,fb_terms=10, fb_docs=100)

# Now we expand the query and see the resulting query
# For RM3, we need to first retrieve a set of "psuedo-relevant documents" then pass them to the expansion model
rm3_qe = bm25 >> rm3_expander
expanded_query = rm3_qe.search(query).iloc[0]["query"] #command will return the expanded query inside a dataframe with one row, access that row (iloc[0])

# Just print the expanded query with term scores
for s in expanded_query.split()[1:]:
  print(s)

# Let's compare to the original query
print("\n" + query)

لبنانيه^0.015987111
7^0.026475038
انتحار^0.193639398
هجوم^0.120000005
مزدوج^0.021354133
مقه^0.190307260
مدين^0.016528953
مقتل^0.047432967
سبع^0.021423412
طرابلس^0.167161152
لبنان^0.179690570

هجوم انتحار مقه طرابلس لبنان


In [ ]:
# After that you can search using the expanded query
expanded_query_formatted = ' '.join(expanded_query.split()[1:])
results_wqe = bm25.search(expanded_query_formatted)

if len(results_wqe)==0:
   print("There are no relevant documents for your selected query.")
else:
  print("   Before Expansion    After Expansion")
  print(pd.concat([results_noqe[['docid','score']][0:5].add_suffix('_1'),
            results_wqe[['docid','score']][0:5].add_suffix('_2')], axis=1).fillna(''))
   #print(results_wqe[['docid','score']][0:5])

#Let's check the tweets text for the top 5 retrieved tweets
full_data[['tweetText']][full_data['docno'].isin(results_wqe['docno'].loc[0:5].tolist())]

   Before Expansion    After Expansion
   docid_1    score_1  docid_2    score_2
0    21590  25.403218    21590  26.398558
1    22944  25.297453    22944  25.871495
2    22646  23.988924    22646  25.705874
3    22692  23.988924    22692  25.705874
4    22438  23.656683    21503  22.278665


,tweetText
21503,لبنان: مقتل سبعة في هجوم انتحاري مزدوج على مقهى http://t.co/v3k1DtIURZ http://t.co/49uqJSdfKS
21590,#لبنان - مقتل 7 في هجوم انتحاري مزدوج على مقهى في #طرابلس #bahrain #lebanon
21665,لبنان: مقتل سبعة في هجوم انتحاري مزدوج على مقهى http://t.co/7N2HRJbKIV
22646,مقتل 7 في هجوم انتحاري على مقهى في لبنان: أدى هجوم انتحاري على مقهى في مدينة طرابلس اللبنانية إلى مقتل سبعة أش... http://t.co/d1zWvCG3dh
22692,مقتل 7 في هجوم انتحاري على مقهى في لبنان: أدى هجوم انتحاري على مقهى في مدينة طرابلس اللبنانية إلى مقتل سبعة أش... http://t.co/uwvvF6V5bD
22944,مقتل سبعة على الأقل في هجوم انتحاري على مقهى في لبنان - أدى هجوم انتحاري على مقهى في مدينة طرابلس اللبنانية إلى ... http://t.co/kRD9oPXlra


### **Retrieval with QE on all queries**
We will now work on the full set of queries and the qrels (relevance judgements) provided with EveTAR.

In [ ]:
# We will run the full pipeline explained above. PyTerrier made it as easy as follows
rm3_pipe = bm25 >> rm3_expander >> bm25
pt.Experiment([bm25, rm3_pipe],
            queriesDF,
            qrels,
            ["map"],
            round={"map" : 4}, #"round" will round the score to 4 digits
            names=["BM25", "BM25+RM3"])

,name,map
0,BM25,0.5966
1,BM25+RM3,0.6378


## **Exercise1**
Study the effect of changing the number of feedback terms and documents on PRF using RM3 for term selection and BM25 as the retrieval model. Use the following ranges:
*   feedback terms (fb_terms): {5,10}
*   feedback documents (fb_docs): {10,30}

**Hints**:
* Don't forget to do all combinations.
* Compare to the performance to original BM25

In [ ]:
# The basic model will give us the initial results.
bm25 = pt.BatchRetrieve(index, wmodel="BM25")

# --> First experience
rm3_t5_d10 = bm25 >> pt.rewrite.RM3(index, fb_terms=5, fb_docs=10) >> bm25

# --> Second experience
rm3_t10_d10 = bm25 >> pt.rewrite.RM3(index, fb_terms=10, fb_docs=10) >> bm25

# --> Third experience
rm3_t5_d30 = bm25 >> pt.rewrite.RM3(index, fb_terms=5, fb_docs=30) >> bm25

# --> Fourth experience
rm3_t10_d30 = bm25 >> pt.rewrite.RM3(index, fb_terms=10, fb_docs=30) >> bm25

/tmp/ipykernel_845/3153955516.py:2: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25 = pt.BatchRetrieve(index, wmodel="BM25")


In [ ]:
# Comparison between the baseline model and the four experiments.
experiment_results = pt.Experiment(
    [bm25, rm3_t5_d10, rm3_t10_d10, rm3_t5_d30, rm3_t10_d30],
    queriesDF,
    qrels,
    ["map"],
    round={"map" : 4},
    names=["Baseline BM25", "RM3 (t=5, d=10)", "RM3 (t=10, d=10)", "RM3 (t=5, d=30)", "RM3 (t=10, d=30)"]
)

experiment_results

,name,map
0,Baseline BM25,0.5966
1,"RM3 (t=5, d=10)",0.6269
2,"RM3 (t=10, d=10)",0.6358
3,"RM3 (t=5, d=30)",0.6276
4,"RM3 (t=10, d=30)",0.6376
